# DISTRIBUTION-AWARE **ITERATIVE** V1 — CANDIDATE-SIDE DIAGNOSTIC

**Select a T4 GPU, then Runtime -> Run all, then leave it.** Roughly three
hours. Everything lands on Drive under `results/distribution_aware_iterative_diagnostic/`
and **nothing else is touched**.

This is the cheap diagnostic frozen by
`docs/iterative_decision_memo_2026-09-06.md`. Its only scientific difference
from `distribution_aware_v1` is that the clustering, the rarity and the quota
are recomputed after each annotation round against a reference grown by what
the round bought — **six rounds of 500 answers, remainder carried**.

* **`distribution_aware_v1` remains NO-GO and is not reopened.** Its outputs
  live in `results/distribution_aware_diagnostic/`; this session writes to a
  different directory, asserted in cell [2/10].
* **The comparators run the same schedule.** With the remainder carried, six
  rounds open the identical images in the identical order for a static
  ranking — asserted on the real code path — so `entropy` and `cost_aware`
  are unchanged by it.
* **A prediction is on the record** (memo §3.1): iteration pushes budget
  toward unvisited clusters, which is *more* spreading, and spreading has
  failed three times. If this fails it is the fourth negative and this D/R/C
  formulation stops.

* **No PROB training. No PROB evaluation. No checkpoint is written.** The one
  detector call is `predict`, which scores candidate images. A test asserts the
  driver cannot reach `bridge.train` or `bridge.evaluate`.
* **Nothing of the frozen benchmark is touched.** Different results directory,
  different workspaces. Seeds 0, 1 and 2 of `random`, `admissibility`,
  `entropy`, `proposed` and `proposed_v2` are neither read nor written.
* **The gates are printed and written to Drive before anything is measured**,
  so the criteria are on the record ahead of the outcome. Once the outcome is
  visible none of them may change: not a threshold, not an aggregation, not the
  clusterer, not `min_cluster_size`, not `R`.
* **REF-T1 is mandatory.** `distribution_aware_v1` measures under-representation
  against the balanced task-1 reference plus what the trajectory has bought.
  Cell [7/10] reuses the frozen export if Drive has it and rebuilds it if not,
  and verifies its manifest SHA-256 either way. It will not fall back to an
  empty reference, because that is a different method.
* **One declared deviation.** A benchmark trajectory scores task *n*'s pool with
  the checkpoint task *n-1* produced for that arm. This trains nothing, so every
  task of every arm is scored with the **t1 anchor**. The detector is held fixed
  and the comparison becomes purely one about selection — which is what a
  candidate-side diagnostic is for — but `entropy` here is entropy under the
  anchor. It applies identically to all three arms.
* `distribution_aware_v1` and `cost_aware` are **development-seed-informed and
  not pre-registered**. Any table reporting them must say so.

**It is resumable.** The detector exports and the DINOv2 matrices are cached on
Drive, so a disconnect costs the image fetches and not the GPU work. Re-open and
Run all again.

**The chain declares one class per task** (`traffic light`, `fire hydrant`,
`stop sign`) and is **not** the published S-OWODB split. No number it produces
may be compared against a published S-OWODB result.

In [ ]:
# [1/10] Parameters and immutable experiment identity
# ============================== PARAMETERS ==============================
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

# Before anything imports torch. Expandable segments let a freed block be
# reused at a different size instead of fragmenting the reserve, which is what
# turns "enough free memory in total" into an OOM — how the ungated coreset
# attempt died. Memory management only; it cannot change a number.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"
OWL_COMMIT = "c0230e1261522b459b09529f6036e9e6700938b2"
PROB_REPOSITORY = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "4c66be1a52cad9360e09c729e9134aba8fe0b531"

DRIVE_ROOT = "/content/drive/MyDrive/OWL"
CHECKPOINT_RELATIVE = "checkpoints/SOWODB/t1.pth"
FEATURES_RELATIVE = "features"
RESULTS_RELATIVE = "results/distribution_aware_iterative_diagnostic"

DATA_ROOT = "/content/data/OWOD"

# THE CANDIDATE-SIDE DIAGNOSTIC for `distribution_aware_iterative_v1`.
#
# Frozen by docs/iterative_decision_memo_2026-09-06.md. The ONLY scientific
# difference from distribution_aware_v1 -- itself a completed NO-GO that this
# session neither reopens nor overwrites -- is that the clustering, the
# rarity and the quota are recomputed after each annotation round against a
# reference grown by what the round bought. Six rounds of 500 answers with
# the remainder carried, a schedule measured to be a no-op for a static
# ranking, so entropy and cost_aware run it and are unchanged by it.
#
# Frozen by docs/distribution_aware_decision_memo_2026-09-06.md. Selection only:
# no PROB training, no PROB evaluation, no checkpoint is written. The one
# detector call is `predict`, which scores candidate images, and a test asserts
# the driver cannot reach `bridge.train` or `bridge.evaluate`.
#
# The arms and the seeds are NOT written here. They are read from the pinned
# module in cell [3/10], because a number typed into a notebook is a number that
# can drift away from the one the frozen gates are applied with.
#
# ONE DECLARED DEVIATION. A benchmark trajectory scores task n's pool with the
# checkpoint task n-1 produced for that arm. This trains nothing, so there are no
# per-arm checkpoints and every task of every arm is scored with the t1 anchor.
# The detector is held fixed and the comparison becomes purely one about
# selection — which is what a candidate-side diagnostic is for — but `entropy`
# here is entropy under the anchor. It applies identically to all three arms.
#
# `distribution_aware_v1` and `cost_aware` are development-seed-informed and NOT
# pre-registered. Seeds 0, 1 and 2 of the frozen benchmark are neither read nor
# written by this session.

SESSION_STARTED = time.monotonic()

# Every scientific constant is READ from the pinned repository, never restated
# here — a number typed into a notebook is a number that can drift away from the
# module the results are actually computed from.
assert len(PROB_COMMIT) == 40 and len(OWL_COMMIT) == 40, "pin full 40-char SHAs"
print("OWL commit :", OWL_COMMIT)
print("PROB commit:", PROB_COMMIT)



In [ ]:
# [2/10] Mount Drive and prove the persistent root is writable
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
_probe = DRIVE / ".benchmark_v1_write_probe"
_probe.write_text("ok", encoding="utf-8")
assert _probe.read_text(encoding="utf-8") == "ok"
_probe.unlink()

FEATURES = DRIVE / FEATURES_RELATIVE
RESULTS = DRIVE / RESULTS_RELATIVE
CHECKPOINT = DRIVE / CHECKPOINT_RELATIVE
RESULTS.mkdir(parents=True, exist_ok=True)

# The frozen benchmark lives next door. Named here so the separation is visible
# in the transcript rather than only in a docstring.
FROZEN = DRIVE / "results" / "full_owod_active_benchmark_v1"
ONE_SHOT = DRIVE / "results" / "distribution_aware_diagnostic"
assert RESULTS not in (FROZEN, ONE_SHOT)
assert FROZEN not in RESULTS.parents and ONE_SHOT not in RESULTS.parents
print("Drive writable:", DRIVE)
print("writes to     :", RESULTS)
print("untouched     :", FROZEN, "(exists)" if FROZEN.exists() else "(absent)")
print("untouched     :", ONE_SHOT,
      "(exists - the completed NO-GO)" if ONE_SHOT.exists() else "(absent)")
print("results ->", RESULTS)


In [ ]:
# [3/10] Pin OWL exactly, install its declared dependencies, import fresh code
def _checked(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)


def _capture(command, **kwargs):
    return _checked(command, capture_output=True, **kwargs).stdout.strip()


def _normalise_git_url(value):
    value = value.strip().removesuffix(".git").rstrip("/")
    if value.startswith("git@github.com:"):
        value = "https://github.com/" + value.split(":", 1)[1]
    return value


def ensure_pinned_checkout(path, repository, commit):
    path = Path(path)
    expected = _normalise_git_url(repository)
    if path.exists():
        assert (path / ".git").is_dir(), f"Refusing non-git path: {path}"
        origin = _normalise_git_url(_capture(["git", "remote", "get-url", "origin"], cwd=path))
        assert origin == expected, f"Refusing unexpected origin at {path}: {origin}"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        _checked(["git", "clone", "--filter=blob:none", "--no-checkout", repository, str(path)])
    _checked(["git", "fetch", "--depth", "1", "origin", commit], cwd=path)
    _checked(["git", "reset", "--hard", commit], cwd=path)
    _checked(["git", "clean", "-fdx"], cwd=path)
    actual = _capture(["git", "rev-parse", "HEAD"], cwd=path)
    assert actual == commit, f"{path}: expected {commit}, got {actual}"
    return path


ROOT = ensure_pinned_checkout(Path("/content/owod-active"), OWL_REPOSITORY, OWL_COMMIT)
_checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
          "-e", f"{ROOT}[plots]"])

for _name in [n for n in sys.modules if n == "owl" or n.startswith("owl.")]:
    del sys.modules[_name]
sys.path.insert(0, str(ROOT))

from owl import bridge, evaluation_subset, metrics, protocol, runner
from owl.active_selection import arms as arm_registry
from owl.active_selection import benchmark as bm
from owl.active_selection import budget as annotation_budget
from owl.active_selection import allocation, coverage, diagnostic, population, semantic

# Named checks, so a stale OWL_COMMIT reports which API it is missing instead of
# failing as an AttributeError four cells later.
_required = {
    "runner.CycleConfig.budget_unit": "budget_unit" in {
        f.name for f in __import__("dataclasses").fields(runner.CycleConfig)},
    "run_chain.selector": "selector" in __import__("inspect").signature(
        runner.run_chain).parameters,
    "bm.check_protocol": hasattr(bm, "check_protocol"),
    "bm.make_selector": hasattr(bm, "make_selector"),
    "bm.cycle_config": hasattr(bm, "cycle_config"),
    "arms.ORDER": hasattr(arm_registry, "ORDER"),
    "coverage.kcenter_greedy": hasattr(coverage, "kcenter_greedy"),
    "population.p2_reference": hasattr(population, "p2_reference"),
    "semantic.cached": hasattr(semantic, "cached"),
    "budget.cost_function": hasattr(annotation_budget, "cost_function"),
    # Two things a symbol check cannot see, both of which a stale pin would
    # silently lack: the ledger column that says what PROB was actually handed,
    # and the fail-closed guard that stops a resumed chain from restarting a
    # task from the anchor and reporting a sequential result it never produced.
    "runner.boxes_trained_on": "boxes_trained_on" in __import__("inspect").getsource(
        runner.run_chain),
    "runner.lineage_guard": "break the checkpoint lineage" in __import__(
        "inspect").getsource(runner.run_chain),
    # Proposed-v2's surface. A stale pin lacking any of these must say which.
    "arms.proposed_v2": "proposed_v2" in arm_registry.ARMS,
    "arms.ranked_positions": hasattr(arm_registry, "ranked_positions"),
    "arms.informative": getattr(
        arm_registry.ARMS.get("proposed_v2"), "informative", False),
    "arms.reference_scope": getattr(
        arm_registry.ARMS.get("proposed_v2"), "reference_scope", "") == "trajectory",
    "bm.KILL_RULE": hasattr(bm, "KILL_RULE"),
    "bm.PROVENANCE": hasattr(bm, "PROVENANCE"),
    "semantic.release": hasattr(semantic, "release"),
    # The 2026-09-06 surface. A stale pin lacking any of these must say which.
    "arms.distribution_aware_v1": "distribution_aware_v1" in arm_registry.ARMS,
    "arms.cost_aware": "cost_aware" in arm_registry.ARMS,
    "arms.ALLOCATED": hasattr(arm_registry, "ALLOCATED"),
    "allocation.distribution_aware_order": hasattr(
        allocation, "distribution_aware_order"),
    "allocation.cost_aware_order": hasattr(allocation, "cost_aware_order"),
    "allocation.min_cluster_size": hasattr(allocation, "min_cluster_size"),
    "diagnostic.GATES": len(getattr(diagnostic, "GATES", ())) == 6,
    "diagnostic.configuration": hasattr(diagnostic, "configuration"),
    "diagnostic.evaluate": hasattr(diagnostic, "evaluate"),
    "diagnostic.method_under_test": hasattr(diagnostic, "method_under_test"),
    "diagnostic.ITERATIVE_ROUNDS": getattr(diagnostic, "ITERATIVE_ROUNDS", 0) == 6,
    "arms.distribution_aware_iterative_v1":
        "distribution_aware_iterative_v1" in arm_registry.ARMS,
    "arms.ITERATIVE": hasattr(arm_registry, "ITERATIVE"),
    "allocation.iterative": hasattr(
        allocation, "distribution_aware_iterative_order"),
    "budget.spend_ranking_in_rounds": hasattr(
        annotation_budget, "spend_ranking_in_rounds"),
}
assert all(_required.values()), {k: v for k, v in _required.items() if not v}
# The registry's *properties*, not a literal copy of it. A hardcoded tuple has
# to be edited by hand every time an arm is added, and when it is not, Run all
# dies in this cell -- which is exactly what happened once the iterative arm was
# registered. What actually matters scientifically is asserted instead: the
# pre-registered arms come first in the declared order, and no arm designed after
# seeing results displaces one of them. `tests/test_active_selection.py` enforces
# the same property in the suite.
assert arm_registry.ORDER[:5] == (
    "random", "admissibility", "proposed", "entropy", "coreset",
), arm_registry.ORDER
assert set(arm_registry.ORDER) == set(arm_registry.ARMS), arm_registry.ORDER
_informed = set(bm.DEVELOPMENT_SEED_INFORMED)
_positions = [i for i, a in enumerate(arm_registry.ORDER) if a in _informed]
_baselines = [i for i, a in enumerate(arm_registry.ORDER) if a not in _informed]
assert min(_positions) > max(_baselines), arm_registry.ORDER
print("registry  :", len(arm_registry.ARMS), "arms;",
      len(_informed), "development-seed-informed, all after the baselines")

# The scientific configuration, read from the pinned module and never retyped.
SESSION_ARMS = diagnostic.ITERATIVE_ARMS
SESSION_SEEDS = diagnostic.DIAGNOSTIC_SEEDS
SESSION_ROUNDS = diagnostic.ITERATIVE_ROUNDS
assert set(SESSION_ARMS) <= set(arm_registry.ARMS), SESSION_ARMS
assert SESSION_ARMS == (
    "entropy", "cost_aware", "distribution_aware_iterative_v1")
assert SESSION_ROUNDS == 6
assert diagnostic.method_under_test(SESSION_ARMS) \
    == "distribution_aware_iterative_v1"
print("arms  :", SESSION_ARMS)
print("seeds :", SESSION_SEEDS, "| rounds:", SESSION_ROUNDS)
assert bm.N_TASKS == 4 and bm.ANSWER_BUDGET_PER_TASK == 3000
print("chain:", [t.name for t in bm.chain()])
print("budget:", bm.ANSWER_BUDGET_PER_TASK, "oracle answers per task,",
      bm.CANDIDATE_IMAGES_PER_TASK, "candidate images")
# HDBSCAN. Colab's sklearn is usually new enough; this is the named failure if
# it is not, rather than an ImportError inside the first clustering call.
import sklearn
from sklearn.cluster import HDBSCAN  # noqa: F401
print("sklearn", sklearn.__version__, "— HDBSCAN available")

OWL_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=ROOT)
print("OWL ready:", OWL_SHA, "from", ROOT)


In [ ]:
# [4/10] PREFLIGHT — is the pinned PROB source actually there?
#
# A Method V3 overnight run died in the next cell on
#
#     git clone --filter=blob:none --no-checkout .../PROB.git   ->  exit status 128
#
# after Drive was mounted and OWL was installed, and a bare 128 does not say
# whether the URL is wrong, the pinned commit is gone, or a shared Colab egress
# address was rate-limited for a minute. Guessing at that is how a frozen
# detector gets quietly swapped for a convenient one, so the question is
# answered here — before pip, before the CUDA kernel build — and the answer
# names the URL and the SHA.
# Runtime -> Run all executes cells in order and this holds automatically. It is
# checked anyway because the failure it replaces is a bare `NameError: name
# 'bridge' is not defined`, which says nothing about what to do — and a Colab
# user who runs one cell to "just check something" gets exactly that.
if "bridge" not in globals():
    raise RuntimeError(
        "cell [3/10] has not run in this kernel, so `bridge` does not exist. "
        "It is the cell that pins OWL and imports the package. Use "
        "Runtime -> Run all rather than running cells individually."
    )

PROB_REMOTE = bridge.verify_remote_commit(PROB_REPOSITORY, PROB_COMMIT)
print("PROB repository :", PROB_REMOTE["repository"], "— reachable")
print("PROB commit     :", PROB_REMOTE["commit"], "— present on the server")
print("PROB branch     :", PROB_REMOTE["branch"], "->", PROB_REMOTE["branch_head"])
print("pin is a ref tip:", PROB_REMOTE["pin_is_ref_tip"],
      "| branch still points at the pin:", PROB_REMOTE["branch_points_at_commit"])
print("probe attempts  :", PROB_REMOTE["attempts_used"])
if not PROB_REMOTE["branch_points_at_commit"]:
    print("NOTE: the branch has moved on since the pin. The pin is honoured "
          "anyway — that is what a pin is for.")


In [ ]:
# [5/10] Pin and validate the reviewed PROB bridge; build the optional CUDA kernel
# An exact local checkout is accepted as-is: origin must be the pinned
# repository and HEAD must be the pinned SHA, so what is accepted is
# byte-identical to what a clone would have produced. That makes a second
# attempt in the same session free, and it is the offline recovery — a mirror of
# that SHA copied in from Drive is scientifically the same run.
_PROB_PATH = Path("/content/PROB")
if bridge.local_checkout_matches(_PROB_PATH, PROB_REPOSITORY, PROB_COMMIT):
    PROB = _PROB_PATH
    print("PROB already checked out at the pinned commit; no network needed.")
else:
    PROB = None
    for _attempt in range(1, 4):
        try:
            PROB = ensure_pinned_checkout(_PROB_PATH, PROB_REPOSITORY, PROB_COMMIT)
            break
        except subprocess.CalledProcessError as _error:
            # A failed clone can leave a partial directory behind, which the next
            # attempt would then treat as an existing checkout. Clear it unless it
            # is a real git repository.
            if not (_PROB_PATH / ".git").is_dir():
                shutil.rmtree(_PROB_PATH, ignore_errors=True)
            print(f"PROB checkout attempt {_attempt}/3 failed: {_error}")
            if _attempt == 3:
                raise
            time.sleep(10)
    assert PROB is not None

# PROB's 2022 requirements file pins packages that have no Python 3.13 wheels
# (notably scikit-image 0.19.2 and pandas 1.5.1). The bridge does not import
# scikit-image, notebook, or ipdb. Install only its runtime imports, without
# replacing Colab's matched torch/torchvision/numpy stack. pycocotools stays
# at PROB's exact 2.0.5 pin: Cython generates the C source omitted by its sdist.
assert sys.version_info[:2] == (3, 13), sys.version
def distribution_version(distribution):
    probe = subprocess.run(
        [sys.executable, "-c",
         f"from importlib.metadata import version; print(version({distribution!r}))"],
        capture_output=True, text=True, check=False)
    return probe.stdout.strip() if probe.returncode == 0 else None


def module_available(module):
    return subprocess.run(
        [sys.executable, "-c", f"import {module}"],
        capture_output=True, text=True, check=False).returncode == 0


PROB_COMPAT_INSTALLED = []
if distribution_version("einops") != "0.5.0" or not module_available("einops"):
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "einops==0.5.0"])
    PROB_COMPAT_INSTALLED.append("einops==0.5.0")

if (distribution_version("pycocotools") != "2.0.5"
        or not module_available("pycocotools")):
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "Cython==3.1.3"])
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--no-build-isolation",
              "--no-deps", "--force-reinstall", "pycocotools==2.0.5"])
    PROB_COMPAT_INSTALLED.append("pycocotools==2.0.5")

# These imports are required transitively by main_open_world/engine. Keep a
# compatible Colab package when one is already importable; install a pinned
# Python-3.13 wheel only when it is absent or broken. WandB is disabled by the
# reviewed bridge and therefore cannot affect training or evaluation.
compatibility_wheels = {
    "wandb": "wandb==0.18.7",
    "pandas": "pandas==2.3.2",
    "seaborn": "seaborn==0.13.2",
    "tqdm": "tqdm==4.67.1",
}
missing_wheels = [spec for module, spec in compatibility_wheels.items()
                  if not module_available(module)]
if missing_wheels:
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              *missing_wheels])
    PROB_COMPAT_INSTALLED.extend(missing_wheels)

assert distribution_version("einops") == "0.5.0" and module_available("einops")
assert (distribution_version("pycocotools") == "2.0.5"
        and module_available("pycocotools"))


def pip_check():
    return subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        capture_output=True, text=True, check=False)


# Current Colab carries IPython metadata that requires Jedi while omitting
# Jedi itself. Repair only that observed metadata conflict, using a universal
# wheel with explicit Python 3.13 support, then require the complete package
# environment to pass the same check used by the final preflight.
bootstrap_package_probe = pip_check()
_package_conflicts = bootstrap_package_probe.stdout + bootstrap_package_probe.stderr
_missing_ipython_jedi = (
    "requires jedi, which is not installed" in _package_conflicts.lower()
    and "ipython " in _package_conflicts.lower()
)
if _missing_ipython_jedi:
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "jedi==0.19.2"])
    PROB_COMPAT_INSTALLED.append("jedi==0.19.2")
    bootstrap_package_probe = pip_check()
if bootstrap_package_probe.returncode != 0:
    print(bootstrap_package_probe.stdout + bootstrap_package_probe.stderr)
    raise RuntimeError("Python package consistency check failed after bootstrap repair")
print("Bootstrap package consistency: PASS")
runtime_probe = subprocess.run(
    [sys.executable, "-c",
     "import numpy, torch, torchvision, scipy, sklearn, PIL, matplotlib, pandas, seaborn, tqdm, wandb; "
     "from einops import rearrange; from pycocotools.coco import COCO; "
     "import main_open_world; from datasets.coco import make_coco_transforms; "
     "from datasets.torchvision_datasets.open_world import OWDetection; "
     "from engine import evaluate; from models import build_model"],
    cwd=PROB, capture_output=True, text=True, check=False)
if runtime_probe.returncode != 0:
    print(runtime_probe.stdout)
    print(runtime_probe.stderr)
    raise RuntimeError("Pinned PROB failed its Python runtime import probe")
print("PROB runtime imports: PASS; installed:", PROB_COMPAT_INSTALLED or "nothing")

# pycocotools 2.0.5 predates NumPy 2.0. Exercise PROB's own evaluator
# wrapper with the two removed aliases it needs, instead of accepting an
# import-only success. The aliases are confined to this fresh subprocess.
coco_smoke_code = r'''
import numpy as np
import torch
if "float" not in np.__dict__:
    np.float = float
if "NPY_OWNDATA" not in np.__dict__:
    np.NPY_OWNDATA = 4
from pycocotools.coco import COCO
from datasets.coco_eval import CocoEvaluator
coco = COCO()
coco.dataset = {
    "info": {}, "licenses": [],
    "images": [{"id": 1, "width": 32, "height": 32}],
    "categories": [{"id": 1, "name": "object", "supercategory": "object"}],
    "annotations": [{"id": 1, "image_id": 1, "category_id": 1,
                     "bbox": [4.0, 5.0, 10.0, 11.0], "area": 110.0, "iscrowd": 0}],
}
coco.createIndex()
evaluator = CocoEvaluator(coco, ("bbox",))
evaluator.update({1: {"boxes": torch.tensor([[4.0, 5.0, 14.0, 16.0]]),
                      "scores": torch.tensor([0.99]), "labels": torch.tensor([1])}})
evaluator.synchronize_between_processes()
evaluator.accumulate()
evaluator.summarize()
assert float(evaluator.coco_eval["bbox"].stats[0]) > 0.99
print("PROB pycocotools COCOeval smoke: PASS")
'''
coco_smoke = subprocess.run([sys.executable, "-c", coco_smoke_code], cwd=PROB,
                            capture_output=True, text=True, check=False)
if coco_smoke.returncode != 0:
    print(coco_smoke.stdout)
    print(coco_smoke.stderr)
    raise RuntimeError("Pinned pycocotools failed PROB's functional COCOeval smoke test")
print(coco_smoke.stdout.splitlines()[-1])


def run_json_probe(code, marker, *, cwd):
    probe = subprocess.run(
        [sys.executable, "-c", code], cwd=cwd,
        capture_output=True, text=True, check=False,
    )
    rows = [line.removeprefix(marker) for line in probe.stdout.splitlines()
            if line.startswith(marker)]
    if not rows:
        return {
            "probe_ok": False,
            "returncode": probe.returncode,
            "error": (probe.stderr or probe.stdout).strip() or "probe produced no result",
        }
    payload = json.loads(rows[-1])
    payload["returncode"] = probe.returncode
    return payload


# Deliberately diagnostic only: unlike PROB, this fresh interpreter does not
# import torch before loading the extension. On Colab that can fail to resolve
# PyTorch shared libraries even when PROB's real import and dispatch work.
raw_msda_probe_code = r"""
import importlib
import json
try:
    importlib.invalidate_caches()
    extension = importlib.import_module("MultiScaleDeformableAttention")
    payload = {"ok": True, "path": getattr(extension, "__file__", None), "error": None}
except BaseException as error:
    payload = {"ok": False, "path": None,
               "error": f"{type(error).__name__}: {error}"}
print("OWOD_RAW_MSDA_PROBE=" + json.dumps(payload, sort_keys=True))
"""
RAW_MSDA = run_json_probe(
    raw_msda_probe_code, "OWOD_RAW_MSDA_PROBE=", cwd=PROB.parent)

# Authoritative pre-build/post-build probe: import through the exact wrapper and
# downstream module used by PROB training. The pinned wrapper imports torch
# before the extension and the downstream module copies this boolean for dispatch.
prob_msda_probe_code = r"""
import importlib
import importlib.metadata
import json
import platform
import sys
from pathlib import Path
import einops
import matplotlib
import numpy
import pandas
import PIL
import pycocotools
import scipy
import sklearn
import torch
import torchvision

after_torch = {"ok": False, "path": None, "error": None}
try:
    importlib.invalidate_caches()
    extension = importlib.import_module("MultiScaleDeformableAttention")
    after_torch = {"ok": True, "path": getattr(extension, "__file__", None), "error": None}
except BaseException as error:
    after_torch["error"] = f"{type(error).__name__}: {error}"

payload = {
    "probe_ok": False,
    "python": platform.python_version(),
    "executable": sys.executable,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torch_cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "pillow": PIL.__version__,
    "matplotlib": matplotlib.__version__,
    "pandas": pandas.__version__,
    "einops": importlib.metadata.version("einops"),
    "pycocotools": importlib.metadata.version("pycocotools"),
    "extension_after_torch": after_torch,
}
try:
    from models.ops.functions import ms_deform_attn_func as msda_func
    from models.ops.modules import ms_deform_attn as msda_module
    expected_root = Path.cwd().resolve()
    wrapper_path = Path(msda_func.__file__).resolve()
    downstream_path = Path(msda_module.__file__).resolve()
    assert wrapper_path.is_relative_to(expected_root), wrapper_path
    assert downstream_path.is_relative_to(expected_root), downstream_path
    available = bool(msda_func.MSDA_AVAILABLE)
    assert bool(msda_module.MSDA_AVAILABLE) == available
    payload.update({
        "probe_ok": True,
        "available": available,
        "backend": "compiled" if available else "PyTorch fallback",
        "wrapper_path": str(wrapper_path),
        "downstream_path": str(downstream_path),
        "extension_path": (getattr(msda_func.MSDA, "__file__", None)
                           if available else None),
        "error": None,
    })
except BaseException as error:
    payload["error"] = f"{type(error).__name__}: {error}"
print("OWOD_PROB_MSDA_PROBE=" + json.dumps(payload, sort_keys=True))
"""


def probe_prob_msda():
    return run_json_probe(
        prob_msda_probe_code, "OWOD_PROB_MSDA_PROBE=", cwd=PROB)


PREBUILD_PROB_MSDA = probe_prob_msda()
_msda_fingerprint = json.dumps({
    "prob": PROB_COMMIT,
    "python": PREBUILD_PROB_MSDA.get("python", sys.version),
    "torch": PREBUILD_PROB_MSDA.get("torch"),
    "torch_cuda": PREBUILD_PROB_MSDA.get("torch_cuda"),
    "gpu": PREBUILD_PROB_MSDA.get("gpu"),
}, sort_keys=True)
MSDA_BUILD_MARKER = (
    PROB.parent / ".owod-active-cache" /
    f"msda-build-{hashlib.sha256(_msda_fingerprint.encode()).hexdigest()[:16]}.json"
)
MSDA_BUILD_ATTEMPTED = False
MSDA_BUILD_RETURN_CODE = None
if PREBUILD_PROB_MSDA.get("available") is not True:
    if MSDA_BUILD_MARKER.is_file():
        print("Skipping a previously built-but-unused MSDA extension for this exact runtime:",
              MSDA_BUILD_MARKER)
    else:
        if not module_available("ninja"):
            _checked([sys.executable, "-m", "pip", "install",
                      "--disable-pip-version-check", "-q", "ninja"])
        MSDA_BUILD_ATTEMPTED = True
        build = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--no-build-isolation", "--no-deps", "--force-reinstall", "."],
            cwd=PROB / "models" / "ops", text=True,
            capture_output=True, check=False,
        )
        MSDA_BUILD_RETURN_CODE = build.returncode
        if build.returncode != 0:
            print("WARNING: optional CUDA extension build failed; the full CUDA smoke "
                  "must prove PROB's PyTorch fallback.")
            print("\n".join((build.stdout + "\n" + build.stderr).splitlines()[-20:]))

POSTBUILD_PROB_MSDA = probe_prob_msda()
print("Raw extension import:", "PASS" if RAW_MSDA.get("ok") else "FAIL")
print("Raw extension path:", RAW_MSDA.get("path") or "unavailable")
if RAW_MSDA.get("error"):
    print("Raw extension error:", RAW_MSDA["error"])
print("PROB MSDA_AVAILABLE:", POSTBUILD_PROB_MSDA.get("available"))
print("PROB wrapper path:", POSTBUILD_PROB_MSDA.get("wrapper_path", "unavailable"))
print("PROB extension path:", POSTBUILD_PROB_MSDA.get("extension_path", "unavailable"))
if RAW_MSDA.get("ok") != POSTBUILD_PROB_MSDA.get("available"):
    print("MSDA diagnostic disagreement explained: the raw probe loads the extension "
          "before torch; pinned PROB imports torch first, then binds the extension "
          "inside models.ops.functions.ms_deform_attn_func. Only PROB's downstream "
          "dispatch is authoritative.")

# Run the real PROB builder and training loss on CUDA. This observes the exact
# MSDeformAttn branch taken by the model and requires that branch to participate
# in forward and backward before any experiment evaluation or training can run.
prob_smoke_code = r"""
import json
import sys
import tempfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
if "bool" not in np.__dict__:
    np.bool = np.bool_
import main_open_world
from datasets.coco import make_coco_transforms
from datasets.open_world_eval import voc_eval
from datasets.torchvision_datasets.open_world import OWDetection
from models import build_model
from models.ops.functions import ms_deform_attn_func as msda_func
from models.ops.modules import ms_deform_attn as msda_module
assert torch.cuda.is_available(), "CUDA is unavailable to the real PROB smoke test"
PROB_MSDA_AVAILABLE = bool(msda_func.MSDA_AVAILABLE)
assert bool(msda_module.MSDA_AVAILABLE) == PROB_MSDA_AVAILABLE
assert Path(msda_func.__file__).resolve().is_relative_to(Path.cwd().resolve())
assert Path(msda_module.__file__).resolve().is_relative_to(Path.cwd().resolve())
dispatch_counts = {"compiled": 0, "fallback": 0}
if PROB_MSDA_AVAILABLE:
    original_apply = msda_module.MSDeformAttnFunction.apply
    class ObservedCompiledDispatch:
        @staticmethod
        def apply(*arguments):
            dispatch_counts["compiled"] += 1
            return original_apply(*arguments)
    msda_module.MSDeformAttnFunction = ObservedCompiledDispatch
else:
    original_fallback = msda_module.ms_deform_attn_core_pytorch
    def observed_fallback(*arguments, **keywords):
        dispatch_counts["fallback"] += 1
        return original_fallback(*arguments, **keywords)
    msda_module.ms_deform_attn_core_pytorch = observed_fallback
args = main_open_world.get_args_parser().parse_args([])
args.device = "cuda"
args.dataset = "OWDETR"
args.PREV_INTRODUCED_CLS = 0
args.CUR_INTRODUCED_CLS = 20
args.num_classes = 81
args.model_type = "prob"
args.wandb_project = ""
args.wandb_name = ""
args.batch_size = 1
args.num_workers = 0
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    (root / "Annotations").mkdir()
    (root / "JPEGImages").mkdir()
    (root / "ImageSets" / "OWDETR").mkdir(parents=True)
    image_id = "000000000001"
    (root / "ImageSets" / "OWDETR" / "smoke_val.txt").write_text(image_id + "\n")
    Image.new("RGB", (32, 32), (0, 0, 0)).save(root / "JPEGImages" / f"{image_id}.jpg")
    annotation = ("<annotation><filename>000000000001.jpg</filename>"
                  "<size><width>32</width><height>32</height><depth>3</depth></size>"
                  "<object><name>aeroplane</name><difficult>0</difficult>"
                  "<bndbox><xmin>1</xmin><ymin>1</ymin><xmax>20</xmax><ymax>20</ymax>"
                  "</bndbox></object></annotation>")
    xml_path = root / "Annotations" / f"{image_id}.xml"
    xml_path.write_text(annotation)
    dataset = OWDetection(args, root, image_set="smoke_val", dataset="OWDETR",
                          transforms=make_coco_transforms("smoke_val"))
    image, target = dataset[0]
    assert image.shape[0] == 3 and target["labels"].tolist() == [0]
    _, _, ap, *_ = voc_eval(
        [f"{image_id} 0.99 1 1 20 20"], [str(xml_path)], [image_id],
        "aeroplane", known_classes=["aeroplane"])
    assert float(ap) > 0.99
model, criterion, postprocessors, _ = build_model(args, mode="prob")
checkpoint_path = Path(sys.argv[1]) if sys.argv[1] else None
if checkpoint_path is not None:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    assert isinstance(checkpoint, dict) and isinstance(checkpoint.get("epoch"), int)
    state = checkpoint.get("model", checkpoint)
    model_state = model.state_dict()
    compatible_keys = [name for name, value in state.items()
                       if name in model_state and torch.is_tensor(value)
                       and value.shape == model_state[name].shape]
    assert compatible_keys, "T1 checkpoint has no compatible detector parameters"
    incompatible = model.load_state_dict(state, strict=False)
    assert torch.equal(model.state_dict()[compatible_keys[0]].cpu(),
                       state[compatible_keys[0]].cpu())
    print("T1 checkpoint parsed:", checkpoint["epoch"], len(compatible_keys),
          len(incompatible.missing_keys), len(incompatible.unexpected_keys))
model.to("cuda").train()
outputs = model([torch.rand(3, 64, 64, device="cuda")])
required = {"pred_logits", "pred_boxes", "pred_obj", "pred_features", "aux_outputs"}
assert required <= set(outputs)
targets = [{"labels": torch.tensor([0], device="cuda"),
            "boxes": torch.tensor([[0.5, 0.5, 0.25, 0.25]], device="cuda")}]
losses = criterion(outputs, targets)
weighted = sum(losses[name] * criterion.weight_dict[name]
               for name in losses if name in criterion.weight_dict)
assert torch.isfinite(weighted)
weighted.backward()
results = postprocessors["bbox"](outputs, torch.tensor([[64, 64]], device="cuda"))
assert len(results) == 1 and torch.isfinite(results[0]["boxes"]).all()
torch.cuda.synchronize()
chosen = "compiled" if PROB_MSDA_AVAILABLE else "PyTorch fallback"
assert dispatch_counts["compiled" if PROB_MSDA_AVAILABLE else "fallback"] > 0
assert dispatch_counts["fallback" if PROB_MSDA_AVAILABLE else "compiled"] == 0
print("OWOD_MSDA_RESULT=" + json.dumps({
    "available": PROB_MSDA_AVAILABLE,
    "backend": chosen,
    "dispatch_counts": dispatch_counts,
}, sort_keys=True))
print("MSDA backend:", chosen)
print("PROB CUDA model/loss/evaluator smoke: PASS")
"""
checkpoint_for_smoke = Path(DRIVE_ROOT) / CHECKPOINT_RELATIVE
smoke_checkpoint = str(checkpoint_for_smoke) if checkpoint_for_smoke.is_file() else ""
prob_smoke = subprocess.run(
    [sys.executable, "-c", prob_smoke_code, smoke_checkpoint],
    cwd=PROB, capture_output=True, text=True, check=False)
if prob_smoke.returncode != 0:
    print(prob_smoke.stdout)
    print(prob_smoke.stderr)
    print("Environment preflight: FAIL")
    raise RuntimeError("Pinned PROB failed its real CUDA model/evaluator smoke test")
_smoke_rows = [line.removeprefix("OWOD_MSDA_RESULT=")
               for line in prob_smoke.stdout.splitlines()
               if line.startswith("OWOD_MSDA_RESULT=")]
if not _smoke_rows:
    print(prob_smoke.stdout)
    print("Environment preflight: FAIL")
    raise RuntimeError("Real PROB smoke passed without an authoritative MSDA result")
MSDA_SMOKE_RESULT = json.loads(_smoke_rows[-1])
PROB_MSDA_AVAILABLE = bool(MSDA_SMOKE_RESULT["available"])
PROB_MSDA_BACKEND = MSDA_SMOKE_RESULT["backend"]
assert PROB_MSDA_BACKEND == ("compiled" if PROB_MSDA_AVAILABLE else "PyTorch fallback")
if (MSDA_BUILD_ATTEMPTED and MSDA_BUILD_RETURN_CODE == 0
        and not PROB_MSDA_AVAILABLE):
    MSDA_BUILD_MARKER.parent.mkdir(parents=True, exist_ok=True)
    MSDA_BUILD_MARKER.write_text(json.dumps({
        "fingerprint": _msda_fingerprint,
        "backend": PROB_MSDA_BACKEND,
        "reason": "extension built but PROB selected and fully verified fallback",
    }, indent=2), encoding="utf-8")
PROB_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=PROB)
assert PROB_SHA == PROB_COMMIT
ENVIRONMENT_PREFLIGHT_OK = True
runtime = POSTBUILD_PROB_MSDA
print("=" * 60)
print("OWOD ENVIRONMENT PREFLIGHT")
print("=" * 60)
for label, value in (
    ("Runtime Python", runtime.get("python")),
    ("Torch", runtime.get("torch")),
    ("Torchvision", runtime.get("torchvision")),
    ("Torch CUDA", runtime.get("torch_cuda")),
    ("CUDA available", runtime.get("cuda_available")),
    ("GPU", runtime.get("gpu")),
    ("NumPy", runtime.get("numpy")),
    ("SciPy", runtime.get("scipy")),
    ("sklearn", runtime.get("sklearn")),
    ("Pillow", runtime.get("pillow")),
    ("matplotlib", runtime.get("matplotlib")),
    ("pandas", runtime.get("pandas")),
    ("einops", runtime.get("einops")),
    ("pycocotools", runtime.get("pycocotools")),
    ("OWL SHA", OWL_SHA),
    ("PROB SHA", PROB_SHA),
    ("Raw MSDA extension import", "PASS" if RAW_MSDA.get("ok") else "FAIL"),
    ("PROB MSDA_AVAILABLE", PROB_MSDA_AVAILABLE),
    ("MSDA backend", PROB_MSDA_BACKEND),
    ("PROB CUDA model/loss/evaluator smoke", "PASS"),
    ("Environment preflight", "PASS"),
):
    print(f"{label}: {value}")
print("=" * 60)



In [ ]:
# [6/10] Build the data root: committed annotations, the shared split, the pixels
def _streamed(command, transcript=None):
    """Run a step, showing its output as it happens, and fail loudly.

    ``capture_output=True`` hides the traceback of the step that failed, which is
    the one thing a 3 a.m. Run all must not do. Everything expensive is streamed
    instead, and the exit code is asserted afterwards.
    """

    print("+", " ".join(map(str, command)))
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in process.stdout:
        print(line.rstrip(), flush=True)
        lines.append(line)
    code = process.wait()
    if transcript is not None:
        Path(transcript).parent.mkdir(parents=True, exist_ok=True)
        with Path(transcript).open("a", encoding="utf-8") as handle:
            handle.write(f"\n=== {time.strftime('%Y-%m-%dT%H:%M:%S')} ===\n")
            handle.writelines(lines)
    assert code == 0, f"{command[1]} exited {code}; its output is above"
    return "".join(lines)


# --annotations-only, and the reason is what this session reads.
#
# Both modes extract the committed archives, which is where the benchmark XML
# for EVERY candidate image comes from, and both write the shared split. The
# only thing full preparation adds is the *evaluation split's pixels* — and this
# diagnostic never evaluates, so fetching them would be preparing something
# nothing reads.
#
# What it does NOT provide either way is the candidate pixels: which 2,000
# images a task scores depends on (seed, task) and on what the arm has already
# bought, so the tool deliberately leaves them to whoever draws the pool. The
# driver materialises them per task through the same materialiser the benchmark
# launcher uses. That is asserted below, not assumed.
_streamed([sys.executable, str(ROOT / "tools" / "prepare_full_owod_benchmark.py"),
           "--data-root", DATA_ROOT, "--annotations-only"])

DATA = Path(DATA_ROOT)
assert (DATA / "Annotations").is_dir(), DATA / "Annotations"
assert (DATA / "ImageSets" / "OWDETR" /
        f"{evaluation_subset.SHARED_TEST_SET}.txt").is_file()

# The candidate annotations specifically — a split file and an Annotations
# directory can both exist while the candidate XML is absent, and PROB reads one
# per image it predicts on.
_index = json.loads(
    (ROOT / "data" / "reference" / "per_image_class_counts.json").read_text())
_sample = sorted(_index)[::4001]
_absent = [i for i in _sample if not (DATA / "Annotations" / f"{i}.xml").is_file()]
assert not _absent, f"candidate annotations missing, e.g. {_absent[:3]}"
print(f"data root ready: {DATA}")
print(f"candidate annotations: sampled {len(_sample)} of {len(_index):,}, all present")

# The driver fetches candidate pixels itself. Prove the path it will use exists.
from tools.materialize_pool_images import materialise as _materialise
print("candidate materialiser:", _materialise.__module__)


In [ ]:
# [7/10] REF-T1 — a MANDATORY scientific prerequisite, not an optional file
#
# The memo defines `distribution_aware_v1`'s reference as **REF-T1 plus every
# image this trajectory has bought**, and R_k is read against it. Without REF-T1
# the reference at t2 is empty, every cluster reads as maximally
# under-represented, and the quota degenerates toward proportional-to-size. That
# is a DIFFERENT METHOD, and it must not be selected by whether an old Drive
# file happens to exist. One protocol, or none.
#
# So: reuse the frozen export if it is there, otherwise rebuild it — and either
# way verify it is bit-for-bit the frozen reference before anything is measured.
from owl import reference_t1 as _ref
from tools.export_ref_t1_features import read as _read_ref

FROZEN_REF_MANIFEST = (
    "a062fc8f4fd43ea52842725aeaa5eccc0e06eab1894b867b248927bd9d2a2a63")
REF_T1 = FEATURES / "ref_t1_dinov2_vitb14_cap1000_v1.npz"
FEATURES.mkdir(parents=True, exist_ok=True)

if not REF_T1.is_file():
    print(f"{REF_T1.name} is absent from Drive. Rebuilding the frozen export.")
    print("This fetches 14,901 task-1 source images and embeds 19,000 crops; "
          "roughly 20 minutes, and it is resumable.")
    # Asserts the frozen REF-T1 manifest BEFORE it downloads anything, so a run
    # that would produce a different reference stops immediately.
    _streamed([sys.executable, str(ROOT / "tools" / "bootstrap_stage2_data.py"),
               "--data-root", DATA_ROOT])
    _streamed([sys.executable, str(ROOT / "tools" / "export_ref_t1_features.py"),
               "--data-root", DATA_ROOT, "--out", str(REF_T1),
               "--per-class-cap", str(_ref.PRIMARY_REF_T1_CAP_PER_CLASS)])

assert REF_T1.is_file(), (
    f"{REF_T1} could not be produced. The diagnostic refuses to run with an "
    "empty reference, because that measures a different method.")

_payload = _read_ref(REF_T1)
_fingerprint = _ref.manifest_fingerprint(_payload["keys"])
assert _fingerprint == FROZEN_REF_MANIFEST, (
    f"REF-T1 at {REF_T1} has manifest {_fingerprint[:16]}, and the frozen "
    f"reference is {FROZEN_REF_MANIFEST[:16]}. This is a different reference "
    "set, so every cosine measured against it would answer a different "
    "question. Refusing.")
assert _payload["embeddings"].shape[0] == 19_000, _payload["embeddings"].shape
print(f"REF-T1 verified: {_payload['embeddings'].shape[0]:,} objects, "
      f"manifest {_fingerprint[:16]}... — the frozen primary reference")

assert CHECKPOINT.is_file(), CHECKPOINT
print(f"t1 anchor  : {CHECKPOINT} ({CHECKPOINT.stat().st_size / 1e6:.0f} MB)")


In [ ]:
# [8/10] THE FROZEN GATES — printed before anything is measured
if "diagnostic" not in globals():
    raise RuntimeError(
        "cell [3/10] has not run in this kernel, so `diagnostic` does not exist. "
        "It is the cell that pins OWL and imports the package. Use "
        "Runtime -> Run all rather than running cells individually.")

_configuration = diagnostic.configuration()
print(json.dumps(_configuration, indent=2))
(RESULTS / "frozen_gates.json").write_text(
    json.dumps(_configuration, indent=2) + "\n", encoding="utf-8")
print("\nwritten to", RESULTS / "frozen_gates.json",
      "— on the record before any oracle value exists")
print()
print("PROVENANCE — this session runs arms that are NOT pre-registered:")
for _line in bm.PROVENANCE:
    print("  *", _line)


In [ ]:
# [9/10] Run the diagnostic. THIS IS THE LONG CELL — roughly three hours.
#
# Where the time goes, measured from the protocol's own cost basis: ~18,000
# candidate images fetched once, 14 detector passes over 28,000 pool-images at
# ~4 min per thousand, and ~144,000 DINOv2 crops for the one arm that needs
# them. The detector pass dominates.
#
# It is resumable in the way that matters: the detector exports and the DINOv2
# matrices are cached on Drive under RESULTS, so a disconnect costs the image
# fetches and not the GPU work. Re-run this cell.
# What is already on disk, before anything is spent. On a first run this is
# empty; on a resumed one it is the hours you are not paying for again. A
# detector export cut off mid-write is reported here and recomputed below —
# PROB writes that file in place, so unlike the DINOv2 cache it cannot be atomic
# and is validated instead.
_streamed([
    sys.executable, str(ROOT / "tools" / "run_distribution_aware_diagnostic.py"),
    "--out", str(RESULTS), "--verify-cache",
])

_streamed([
    sys.executable, str(ROOT / "tools" / "run_distribution_aware_diagnostic.py"),
    "--prob-root", str(PROB),
    "--data-root", DATA_ROOT,
    "--checkpoint", str(CHECKPOINT),
    "--ref-t1", str(REF_T1),
    "--out", str(RESULTS),
    "--seeds", *[str(s) for s in SESSION_SEEDS],
    "--arms", *SESSION_ARMS,
    "--rounds", str(SESSION_ROUNDS),
], transcript=RESULTS / "diagnostic_log.txt")


In [ ]:
# [10/10] The verdict, read back from what was written
_verdict = json.loads((RESULTS / "verdict.json").read_text(encoding="utf-8"))
for _gate in _verdict["gates"]:
    print(f"{_gate['verdict']:>4}  {_gate['gate']:<32} "
          f"threshold {_gate['threshold']:<6} measured {_gate['measured_per_seed']}")
print()
print("VERDICT:", _verdict["verdict"])
if _verdict["failed"]:
    print("failed :", ", ".join(_verdict["failed"]))
    print("\nThe memo's rule: preserve as a negative candidate-side result. Do "
          "not train, do not tune, do not re-choose the clusterer, do not "
          "soften a threshold. Report which criterion failed and by how much.")
else:
    print("\nGO. Freeze the method, then run the minimal downstream experiment "
          "— distribution_aware_v1 AND cost_aware, seed 0, ~4.5 T4-hours. Do "
          "not launch it from this notebook.")
print(f"\nelapsed {(time.monotonic() - SESSION_STARTED) / 60:.1f} min")
print("rows    :", RESULTS / "diagnostic_rows.csv")
print("gates   :", RESULTS / "frozen_gates.json")
print("log     :", RESULTS / "diagnostic_log.txt")
